# RetailMind AI — Customer Churn Engine
## Consolidated project: Days 1–4

RetailMind AI is being built as one evolving retail analytics platform. This notebook develops its first churn-prediction capability in four increments:

1. **Day 1 — Baseline:** inspect the Telco dataset, prepare features, train a baseline model, evaluate it, and save a reusable inference pipeline.
2. **Day 2 — Model challengers:** compare Decision Tree, Random Forest, and SVM using the same data split and a consistent validation strategy.
3. **Day 3 — Optimisation:** tune candidate models with cross-validation, inspect learning curves, and select a champion using training-only validation results.
4. **Day 4 — Feature improvement and ensembles:** add interpretable customer features, apply feature selection, and compare ensemble models.

**Business objective:** identify customers likely to churn so the retention team can prioritise outreach.

**Positive class:** `Churn = Yes` (encoded as `1`). Recall for this class is important because false negatives represent customers who churn but were not flagged.

> Run the notebook from top to bottom the first time. Each day builds on objects created earlier. The notebook saves fitted pipelines and reports into the project folders.

## 0. Project setup

### Recommended folder structure

Create or extract the project so that it looks like this:

```text
RetailMind_AI/
│
├── data/
│   └── Telco-Customer-Churn.csv
│
├── notebooks/
│   └── RetailMind_Churn_Days_1_to_4.ipynb
│
├── models/                 # Saved fitted pipelines are written here
├── reports/                # Metrics tables and learning-curve output
├── src/                    # Later: reusable application/service code
├── README.md
├── requirements.txt
└── .gitignore
```

### Dataset

Download **Telco Customer Churn** from Kaggle:

https://www.kaggle.com/datasets/blastchar/telco-customer-churn

Place the CSV in `data/`. The notebook accepts either common filename:
- `Telco-Customer-Churn.csv`
- `WA_Fn-UseC_-Telco-Customer-Churn.csv`

### Run in VS Code

1. Open the **RetailMind_AI** folder in VS Code (not just the notebook file).
2. Install the **Python** and **Jupyter** extensions if they are not already installed.
3. Open a terminal in the project folder and create/activate a virtual environment if desired.
4. Install dependencies from `requirements.txt`.
5. Open `notebooks/RetailMind_Churn_Days_1_to_4.ipynb`.
6. Select the Python kernel associated with the environment where the dependencies were installed.
7. Run cells individually with **Run Cell** or `Shift + Enter`. Use **Run All** only after the dataset path and kernel are correct.

If the notebook is moved outside the `notebooks/` folder, update `PROJECT_ROOT` in the next cell.

In [ ]:
# Run this cell once after opening the notebook.
# It checks the project folders and locates the dataset.

from pathlib import Path
import sys
import os
import warnings

# The notebook is expected to live in RetailMind_Project/notebooks/.
# In VS Code, open the RetailMind_Project folder as the workspace.
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "notebooks").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    # Fallback for environments where the current directory is not obvious.
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

for folder in [DATA_DIR, MODEL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    DATA_DIR / "Telco-Customer-Churn.csv",
    DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv",
    DATA_DIR / "telco_customer_churn.csv",
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)

print("Project root:", PROJECT_ROOT)
print("Dataset folder:", DATA_DIR)
print("Models folder:", MODEL_DIR)
print("Reports folder:", REPORT_DIR)

if DATA_PATH is None:
    print("\nDataset not found yet. Download the Kaggle CSV and place it in data/.")
else:
    print("\nDataset found:", DATA_PATH.name)

## 0.1 Install and import the required libraries

The core workflow uses `pandas`, `scikit-learn`, and `joblib`. `matplotlib` is used for the Day 3 learning curve. `xgboost` is optional and is only needed for the XGBoost experiment on Day 4.

If packages are missing, install them in the VS Code terminal using:

```bash
python -m pip install -r requirements.txt
```

In [ ]:
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from src.day4_ensemble import add_customer_features, make_day4_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
    learning_curve,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
TARGET = "Churn"

print("Imports completed.")

# Day 1 — Establish the baseline churn pipeline

**Goal:** create the first working churn model and save the entire preprocessing-plus-model workflow.

The Telco dataset contains both numeric and categorical columns. The preprocessing workflow will:
- convert `TotalCharges` to numeric (blank values become missing values);
- impute missing numeric values with the median;
- scale numeric columns;
- impute missing categorical values with the most frequent value;
- one-hot encode categorical columns.

These transformations are placed inside a `Pipeline` and `ColumnTransformer`. This is important: the transformations are fitted on training data only, which prevents information from the test set leaking into preprocessing.

### 1.1 Load and inspect the CSV

In [ ]:
# Locate the CSV again in case the setup cell was run before the file was copied.
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Place the Kaggle CSV in the project's data/ folder. "
        "Expected Telco-Customer-Churn.csv, WA_Fn-UseC_-Telco-Customer-Churn.csv, or telco_customer_churn.csv."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn data types:")
display(df.dtypes)
print("\nChurn distribution:")
display(df[TARGET].value_counts(dropna=False))
print("\nChurn proportions:")
display(df[TARGET].value_counts(normalize=True).round(3))

### 1.2 Clean the target and prepare features

In [ ]:
# TotalCharges looks numeric but may contain blank strings in the original CSV.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# CustomerID is an identifier, not a behavioural predictor.
df = df.drop(columns=["customerID"], errors="ignore")

# Encode the target: No = 0, Yes = 1.
df[TARGET] = df[TARGET].map({"No": 0, "Yes": 1})

if df[TARGET].isna().any():
    raise ValueError("Unexpected target labels. Expected only 'Yes' and 'No'.")

X = df.drop(columns=TARGET)
y = df[TARGET].astype(int)

print("Feature matrix:", X.shape)
print("Target vector:", y.shape)
print("\nMissing values by feature (non-zero only):")
display(X.isna().sum()[X.isna().sum() > 0])

### 1.3 Create one fixed, stratified train/test split

In [ ]:
# Keep the same split for all four days so that comparisons are consistent.
# Stratification preserves approximately the same churn proportion in both sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)
print("\nChurn rate — train:", round(y_train.mean(), 3))
print("Churn rate — test: ", round(y_test.mean(), 3))

### 1.4 Build reusable preprocessing

In [ ]:
def make_preprocessor(input_frame):
    """Create numeric and categorical preprocessing based on input columns."""
    numeric_columns = input_frame.select_dtypes(include="number").columns.tolist()
    categorical_columns = input_frame.select_dtypes(exclude="number").columns.tolist()

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer(transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ])


def make_model_pipeline(input_frame, estimator, use_feature_selection=False):
    """Package preprocessing and an estimator into one fitted/persistable object."""
    steps = [("preprocess", make_preprocessor(input_frame))]

    if use_feature_selection:
        # k is set later based on the transformed feature count.
        steps.append(("feature_selection", SelectKBest(score_func=f_classif, k=30)))

    steps.append(("model", estimator))
    return Pipeline(steps)


print("Preprocessing and pipeline helper functions are ready.")

### 1.5 Train the Day 1 baseline model

In [ ]:
# Logistic Regression is the Day 1 baseline.
# class_weight='balanced' asks the model to account for the unequal class counts.
baseline_pipeline = make_model_pipeline(
    X_train,
    LogisticRegression(
        max_iter=1500,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
)

baseline_pipeline.fit(X_train, y_train)
print("Day 1 baseline pipeline fitted.")

### 1.6 Evaluate the baseline

In [ ]:
def evaluate_classifier(model_name, fitted_pipeline, features, target, show_details=True):
    """Calculate consistent metrics. Churn=Yes (1) is the positive class."""
    predictions = fitted_pipeline.predict(features)

    if hasattr(fitted_pipeline, "predict_proba"):
        continuous_scores = fitted_pipeline.predict_proba(features)[:, 1]
    else:
        continuous_scores = fitted_pipeline.decision_function(features)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(target, predictions),
        "Precision": precision_score(target, predictions, zero_division=0),
        "Recall": recall_score(target, predictions, zero_division=0),
        "F1": f1_score(target, predictions, zero_division=0),
        "ROC_AUC": roc_auc_score(target, continuous_scores),
    }

    if show_details:
        display(pd.DataFrame([metrics]).set_index("Model"))
        print("Confusion matrix [[TN, FP], [FN, TP]]:")
        print(confusion_matrix(target, predictions))
        print("\nClassification report:")
        print(classification_report(
            target,
            predictions,
            target_names=["No Churn", "Churn"],
            zero_division=0,
        ))

    return metrics


day1_metrics = evaluate_classifier(
    "Day 1 — Logistic Regression baseline",
    baseline_pipeline,
    X_test,
    y_test,
)

### 1.7 Persist the baseline pipeline

In [ ]:
# Save the fitted preprocessing + classifier together.
day1_model_path = MODEL_DIR / "churn_day1_baseline.joblib"
joblib.dump(baseline_pipeline, day1_model_path)

# A small inference helper: new customer records go in; labels and probabilities come out.
def predict_churn(customer_rows, fitted_model=baseline_pipeline):
    """Predict churn label and churn probability for a DataFrame of customer records."""
    churn_labels = fitted_model.predict(customer_rows)
    churn_probabilities = fitted_model.predict_proba(customer_rows)[:, 1]

    return pd.DataFrame({
        "predicted_churn": churn_labels,
        "churn_probability": churn_probabilities,
    }, index=customer_rows.index)


print("Saved:", day1_model_path)
display(predict_churn(X_test.head()))

### Day 1 checkpoint

- Why is `customerID` removed?
- Why is `TotalCharges` converted to numeric?
- Why does the scaler live inside the pipeline rather than being fitted on the complete dataset?
- What does the churn recall value tell the retention team?

**Milestone 1:** the first baseline is saved as `models/churn_day1_baseline.joblib`.

# Day 2 — Compare candidate models

**Goal:** compare Decision Tree, Random Forest, and SVM against the Day 1 baseline.

All candidates use the same preprocessing logic and the same train/test split. We will use cross-validation on the training set to choose a champion. The test set is used to report final, comparable metrics—not to choose the winner.

### 2.1 Define the candidate models

In [ ]:
day2_candidates = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        class_weight="balanced",
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "SVM": SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
}

print("Candidate models:", list(day2_candidates))

### 2.2 Compare candidates using training-only cross-validation

In [ ]:
cv_strategy = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE,
)

day2_fitted = {}
day2_rows = []

for model_name, estimator in day2_candidates.items():
    candidate_pipeline = make_model_pipeline(X_train, estimator)

    # Cross-validation is run only on the training portion.
    cv_recall = cross_val_score(
        candidate_pipeline,
        X_train,
        y_train,
        scoring="recall",
        cv=cv_strategy,
        n_jobs=-1,
    ).mean()

    candidate_pipeline.fit(X_train, y_train)
    day2_fitted[model_name] = candidate_pipeline

    test_metrics = evaluate_classifier(
        f"Day 2 — {model_name}",
        candidate_pipeline,
        X_test,
        y_test,
        show_details=False,
    )
    test_metrics["CV_Recall"] = cv_recall
    day2_rows.append(test_metrics)

day2_comparison = pd.DataFrame(day2_rows).sort_values(
    "CV_Recall", ascending=False
)
display(day2_comparison)
day2_comparison.to_csv(REPORT_DIR / "day2_model_comparison.csv", index=False)

### 2.3 Promote the Day 2 champion

In [ ]:
# Champion selection uses cross-validation recall, not test-set recall.
day2_champion_name = day2_comparison.iloc[0]["Model"].replace("Day 2 — ", "")
day2_champion = day2_fitted[day2_champion_name]

day2_champion_path = MODEL_DIR / "churn_day2_champion.joblib"
joblib.dump(day2_champion, day2_champion_path)

print("Day 2 champion:", day2_champion_name)
print("Selection basis: highest mean cross-validation recall on training data.")
print("Saved:", day2_champion_path)

### Day 2 checkpoint

- Which candidate has the highest cross-validation recall?
- Does the same candidate also lead on precision or ROC-AUC?
- Why should we not choose the champion based on the hold-out test score?

**Milestone 2:** a champion candidate is saved as `models/churn_day2_champion.joblib`. The Day 1 baseline remains available for comparison.

# Day 3 — Hyperparameter tuning and generalisation

**Goal:** tune the candidate models systematically and examine whether they are underfitting or overfitting.

`GridSearchCV` tries the parameter combinations specified below. Each combination is evaluated using stratified cross-validation on the training data. The best estimator is refitted on the full training set.

### 3.1 Define compact parameter grids

In [ ]:
day3_search_spaces = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        {
            "model__max_depth": [4, 8, None],
            "model__min_samples_leaf": [1, 5],
        },
    ),
    "Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        ),
        {
            "model__n_estimators": [150, 250],
            "model__max_depth": [None, 10],
            "model__min_samples_leaf": [1, 3],
        },
    ),
    "SVM": (
        SVC(kernel="rbf", probability=True, class_weight="balanced"),
        {
            "model__C": [0.5, 1, 3],
            "model__gamma": ["scale", "auto"],
        },
    ),
}

print("Search spaces defined.")

### 3.2 Run GridSearchCV

In [ ]:
day3_tuned_models = {}
day3_rows = []

for model_name, (estimator, parameter_grid) in day3_search_spaces.items():
    print(f"Running grid search for {model_name} ...")

    search_pipeline = make_model_pipeline(X_train, estimator)

    search = GridSearchCV(
        estimator=search_pipeline,
        param_grid=parameter_grid,
        scoring="recall",
        cv=cv_strategy,
        n_jobs=-1,
        refit=True,
    )

    search.fit(X_train, y_train)
    day3_tuned_models[model_name] = search.best_estimator_

    day3_rows.append({
        "Model": model_name,
        "Best_CV_Recall": search.best_score_,
        "Best_Parameters": search.best_params_,
    })

day3_cv_results = pd.DataFrame(day3_rows).sort_values(
    "Best_CV_Recall", ascending=False
)
display(day3_cv_results)
day3_cv_results.to_csv(REPORT_DIR / "day3_cv_tuning_results.csv", index=False)

### 3.3 Select and evaluate the tuned champion

In [ ]:
# Choose the model using training cross-validation results.
day3_champion_name = day3_cv_results.iloc[0]["Model"]
day3_champion = day3_tuned_models[day3_champion_name]

day3_champion_path = MODEL_DIR / "churn_day3_tuned.joblib"
joblib.dump(day3_champion, day3_champion_path)

print("Day 3 champion:", day3_champion_name)
print("Best CV recall:", round(day3_cv_results.iloc[0]["Best_CV_Recall"], 4))
print("Saved:", day3_champion_path)

day3_metrics = evaluate_classifier(
    f"Day 3 — Tuned {day3_champion_name}",
    day3_champion,
    X_test,
    y_test,
)

### 3.4 Inspect the learning curve

In [ ]:
# Training recall and validation recall help diagnose generalisation.
# A large gap can indicate overfitting; both scores being low can indicate underfitting.
train_sizes, train_scores, validation_scores = learning_curve(
    day3_champion,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring="recall",
    train_sizes=np.linspace(0.5, 1.0, 3),
    n_jobs=-1,
)

learning_curve_results = pd.DataFrame({
    "Training_Size": train_sizes,
    "Mean_Training_Recall": train_scores.mean(axis=1),
    "Mean_CV_Recall": validation_scores.mean(axis=1),
})
display(learning_curve_results)
learning_curve_results.to_csv(REPORT_DIR / "day3_learning_curve.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(
    train_sizes,
    train_scores.mean(axis=1),
    marker="o",
    label="Training recall",
)
plt.plot(
    train_sizes,
    validation_scores.mean(axis=1),
    marker="o",
    label="Cross-validation recall",
)
plt.xlabel("Number of training examples")
plt.ylabel("Recall for Churn = Yes")
plt.title("Day 3 — Learning curve")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(REPORT_DIR / "day3_learning_curve.png", dpi=140)
plt.show()

### Day 3 checkpoint

- Which parameters were selected for the champion?
- Is training recall much higher than cross-validation recall?
- Does validation recall improve as the training set grows?
- What might you try next if both training and validation recall are low?

**Milestone 3:** the tuned champion is saved as `models/churn_day3_tuned.joblib`, with CV results and learning-curve outputs in `reports/`.

# Day 4 — Feature engineering, feature selection and ensembles

**Goal:** add a small set of business-interpretable features and compare ensemble models against the tuned champion.

The new features are created from existing columns:
- `tenure_group`: broad customer-tenure bands;
- `average_charge_per_month`: total charges divided by tenure;
- `is_month_to_month`: whether the contract is month-to-month;
- `uses_electronic_check`: whether the customer uses electronic check.

Feature selection is fitted inside the pipeline. This ensures the selected features are determined independently within each training fold.

### 4.1 Create additional customer features

In [ ]:
X_train_day4 = add_customer_features(X_train)
X_test_day4 = add_customer_features(X_test)

print("Original feature count:", X_train.shape[1])
print("Feature count after engineering:", X_train_day4.shape[1])
display(X_train_day4.head())


### 4.2 Build a feature-selection pipeline

In [ ]:
# The reusable pipeline applies feature engineering, preprocessing, and
# feature selection within each training fold and during inference.
print("Day 4 pipeline helper is ready.")


### 4.3 Define ensemble candidates

In [ ]:
day4_candidates = {
    "Random Forest + SelectKBest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

# XGBoost is optional. The Random Forest experiment still runs if it is absent.
try:
    from xgboost import XGBClassifier

    day4_candidates["XGBoost + SelectKBest"] = XGBClassifier(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    print("XGBoost is available.")
except ImportError:
    print("XGBoost is not installed. Random Forest will run; install xgboost to add the boosting candidate.")

print("Candidates:", list(day4_candidates))

### 4.4 Compare ensembles using cross-validation

In [ ]:
day4_fitted = {}
day4_rows = []

for model_name, estimator in day4_candidates.items():
    print(f"Cross-validating {model_name} ...")
    candidate_pipeline = make_day4_pipeline(X_train, estimator)

    cv_recall_scores = cross_val_score(
        candidate_pipeline,
        X_train,
        y_train,
        scoring="recall",
        cv=cv_strategy,
        n_jobs=-1,
    )

    # Fit the final candidate on all training rows after CV evaluation.
    candidate_pipeline.fit(X_train, y_train)
    day4_fitted[model_name] = candidate_pipeline

    test_metrics = evaluate_classifier(
        f"Day 4 — {model_name}",
        candidate_pipeline,
        X_test,
        y_test,
        show_details=False,
    )
    test_metrics["CV_Recall_Mean"] = cv_recall_scores.mean()
    test_metrics["CV_Recall_Std"] = cv_recall_scores.std()
    day4_rows.append(test_metrics)

day4_comparison = pd.DataFrame(day4_rows).sort_values(
    "CV_Recall_Mean", ascending=False
)
display(day4_comparison)
day4_comparison.to_csv(REPORT_DIR / "day4_ensemble_comparison.csv", index=False)

### 4.5 Compare the Day 4 candidates with the Day 3 champion

In [ ]:
# The Day 3 champion uses the original input columns.
# Day 4 pipelines engineer features internally from the same raw rows.
# Both are evaluated on the same held-out customer rows.
day3_reference_metrics = evaluate_classifier(
    f"Day 3 champion — {day3_champion_name}",
    day3_champion,
    X_test,
    y_test,
    show_details=False,
)

comparison_rows = [{
    **day3_reference_metrics,
    "CV_Recall_Mean": np.nan,
    "CV_Recall_Std": np.nan,
}]
comparison_rows.extend(day4_rows)

day4_vs_day3 = pd.DataFrame(comparison_rows)
display(day4_vs_day3)
day4_vs_day3.to_csv(REPORT_DIR / "day4_vs_day3_comparison.csv", index=False)

### 4.6 Select and persist the Day 4 champion

In [ ]:
# Champion promotion is based on training-only cross-validation recall.
day4_champion_name = day4_comparison.iloc[0]["Model"]
day4_champion = day4_fitted[day4_champion_name]

day4_champion_path = MODEL_DIR / "churn_day4_champion.joblib"
joblib.dump(day4_champion, day4_champion_path)

print("Day 4 champion:", day4_champion_name)
print("Mean CV recall:", round(day4_comparison.iloc[0]["CV_Recall_Mean"], 4))
print("Saved:", day4_champion_path)

# Final held-out test evaluation for the promoted Day 4 model.
day4_final_metrics = evaluate_classifier(
    f"Day 4 champion — {day4_champion_name}",
    day4_champion,
    X_test,
    y_test,
)

### Day 4 checkpoint

- Which engineered feature might be useful to the retention team, and why?
- Did feature selection improve the result compared with the Day 3 champion?
- Is the Day 4 champion selected using cross-validation rather than test-set performance?
- What additional checks would be needed before replacing a model in a live service?

**Milestone 4:** the Day 4 champion is saved as `models/churn_day4_champion.joblib`.

# Final project summary

The notebook has now produced a sequence of model artifacts:

| Increment | Artifact | Purpose |
|---|---|---|
| Day 1 | `churn_day1_baseline.joblib` | Initial baseline pipeline |
| Day 2 | `churn_day2_champion.joblib` | Best Day 2 candidate by CV recall |
| Day 3 | `churn_day3_tuned.joblib` | Tuned champion selected by CV recall |
| Day 4 | `churn_day4_champion.joblib` | Feature-engineered ensemble champion |

The comparison CSV files and learning-curve outputs are stored in `reports/`.

## What champion means here

A **candidate** is a model being evaluated. The **champion** is the current preferred model according to the agreed selection metric—in this project, mean cross-validation recall for the churn class. A new candidate can replace the champion only when its validation evidence supports the change.

The hold-out test set is kept separate from model selection. It is used to report final performance after a model has been selected.

## What this project covers—and what comes next

This project establishes ML engineering foundations: reusable preprocessing pipelines, repeatable evaluation, cross-validation, model comparison, tuning, feature engineering, and saved model artifacts.

It is not yet a complete production MLOps platform. Later increments can add experiment tracking, a model registry, automated tests, API deployment, CI/CD, monitoring, drift detection, and controlled retraining.

## Git workflow

From the RetailMind project root:

```bash
git add notebooks/RetailMind_Churn_Days_1_to_4.ipynb README.md requirements.txt
git add reports/
git commit -m "Build churn prediction pipeline through Day 4"
git push
```

Avoid committing the raw Kaggle dataset unless repository policy and dataset licensing explicitly permit it. Model artifacts can also be large; decide whether to track them in Git or use a model-artifact store.